In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode != 0:
    raise RuntimeError("❌ Chưa bật GPU! Vào Runtime → Change runtime type → T4 GPU")
print(result.stdout.split('\n')[0])
print("✅ GPU OK")

# Cập nhật và cài đặt các thư viện cần thiết
!pip install -q -U transformers accelerate bitsandbytes peft librosa datasets evaluate jiwer

In [ ]:
import os, json, torch, librosa, numpy as np
import pandas as pd
from datasets import load_dataset
from google.colab import drive
import evaluate
import jiwer

# Mount Google Drive để nạp checkpoint và xuất báo cáo kết quả
drive.mount('/content/drive')

MODEL_ID = "openai/whisper-large-v3"
CHECKPOINTS = {
    "Part_1": "/content/drive/MyDrive/ASR_Continual_Learning/checkpoint_part_1",
    "Part_2": "/content/drive/MyDrive/ASR_Continual_Learning/checkpoint_part_2",
    "Part_3": "/content/drive/MyDrive/ASR_Continual_Learning/checkpoint_part_3",
    "Part_4": "/content/drive/MyDrive/ASR_Continual_Learning/checkpoint_part_4",
    "Part_5": "/content/drive/MyDrive/ASR_Continual_Learning/checkpoint_part_5",
}

# ── Load dataset theo cơ chế Streaming (Tối ưu tốc độ & RAM) ──────────────────
print("📥 Đang kết nối tới dataset LibriSpeech (Streaming Mode)...")
raw_test_stream = load_dataset("openslr/librispeech_asr", "other", split="test", streaming=True)

print("✂️ Đang trích xuất nhanh 5 Parts (200 mẫu/part - Tổng cộng 1000 mẫu)...")
# Sử dụng cơ chế phân đoạn liên tục bằng skip và take để tránh load thừa dữ liệu
eval_datasets = {
    "Part_1": list(raw_test_stream.take(200)),
    "Part_2": list(raw_test_stream.skip(200).take(200)),
    "Part_3": list(raw_test_stream.skip(400).take(200)),
    "Part_4": list(raw_test_stream.skip(600).take(200)),
    "Part_5": list(raw_test_stream.skip(800).take(200)),
}

NORM = jiwer.Compose([
    jiwer.ToLowerCase(),
    jiwer.RemovePunctuation(),
    jiwer.RemoveMultipleSpaces(),
    jiwer.Strip(),
])

wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

print(f"✅ Khởi tạo thành công! Chi tiết tập dữ liệu đánh giá:")
for part_name, data in eval_datasets.items():
    print(f"  - {part_name}: {len(data)} mẫu dữ liệu.")

In [ ]:
from transformers import WhisperProcessor, WhisperForConditionalGeneration, BitsAndBytesConfig, pipeline
from peft import PeftModel

def evaluate_model_on_dataset(checkpoint_path, data_list):
    """
    Hàm load model+adapter và chạy inference song song theo lô (Batch),
    trả về mảng chứa kết quả dự đoán đã được chuẩn hóa.
    Hỗ trợ cả MODEL_ID (base model) và đường dẫn checkpoint LoRA.
    """
    if checkpoint_path != "openai/whisper-large-v3" and not os.path.exists(checkpoint_path):
        print(f"⚠️ Không tìm thấy checkpoint tại: {checkpoint_path}. Gán kết quả bằng NaN.")
        return None, None
        
    print(f"\n🔄 Đang nạp Processor & Model từ: {checkpoint_path}")
    processor = WhisperProcessor.from_pretrained(checkpoint_path if checkpoint_path != "openai/whisper-large-v3" else "openai/whisper-large-v3")
    forced_decoder_ids = processor.get_decoder_prompt_ids(language="en", task="transcribe")
    
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    
    base_model = WhisperForConditionalGeneration.from_pretrained(
        "openai/whisper-large-v3",
        quantization_config=bnb_config,
        device_map="auto"
    )
    
    if checkpoint_path != "openai/whisper-large-v3":
        print(f"🔗 Đang gộp LoRA adapter từ: {checkpoint_path}")
        model_with_lora = PeftModel.from_pretrained(base_model, checkpoint_path)
        model_tuned = model_with_lora.merge_and_unload()
        del model_with_lora
    else:
        print("🍦 Sử dụng Base Model nguyên bản (Không nạp adapter LoRA)")
        model_tuned = base_model
        
    model_tuned.config.forced_decoder_ids = forced_decoder_ids
    model_tuned.eval()
    
    # Khởi tạo Pipeline tối ưu cho GPU
    asr_pipe = pipeline(
        task="automatic-speech-recognition",
        model=model_tuned,
        tokenizer=processor.tokenizer,
        feature_extractor=processor.feature_extractor,
        chunk_length_s=30,
        stride_length_s=5,
        generate_kwargs={
            "language": "en",
            "task": "transcribe",
            "forced_decoder_ids": forced_decoder_ids,
            "suppress_tokens": [],
            "no_repeat_ngram_size": 3,
            "repetition_penalty": 1.2,
            "temperature": 0.0,
            "num_beams": 1,
        },
        device_map="auto",
        batch_size=16
    )
    
    predictions = []
    references = []
    
    print(f"🚀 Đang tính toán inference trên {len(data_list)} mẫu...")
    audio_inputs = []
    raw_references = []
    for item in data_list:
        arr = item["audio"]["array"]
        sr = item["audio"]["sampling_rate"]
        if sr != 16000:
            arr = librosa.resample(arr, orig_sr=sr, target_sr=16000)
        audio_inputs.append(arr)
        raw_references.append(item["text"])
        
    # Chạy pipeline song song qua danh sách audio
    outputs = asr_pipe(audio_inputs)
    
    # Áp dụng bộ chuẩn hóa NORM trực tiếp lên kết quả đầu ra và nhãn gốc
    predictions = [NORM(out["text"]) for out in outputs]
    references = [NORM(ref) for ref in raw_references]
    
    # Giải phóng hoàn toàn VRAM sau mỗi vòng để tránh tràn bộ nhớ (OOM)
    del asr_pipe, model_tuned, base_model
    torch.cuda.empty_cache()
    
    return predictions, references

In [ ]:
parts = ["Part_1", "Part_2", "Part_3", "Part_4", "Part_5"]
N = len(parts)

# Khởi tạo ma trận trống 6x5 (6 hàng: Base Model + 5 Model Parts, 5 cột: 5 Test Parts)
# Hàng 0: Base Model
# Hàng 1..5: Model Part 1..5
wer_matrix = np.full((N + 1, N), np.nan)
cer_matrix = np.full((N + 1, N), np.nan)

# 1. Đánh giá Base Model trên cả 5 Test Parts
print("🕵️‍♂️ BẮT ĐẦU ĐÁNH GIÁ BASE MODEL (openai/whisper-large-v3)")
print("="*65)
for col_idx in range(N):
    data_part = parts[col_idx]
    print(f"\n📊 Đánh giá [Base Model] ↔ [Dữ liệu: {data_part}]")
    preds, refs = evaluate_model_on_dataset(MODEL_ID, eval_datasets[data_part])
    if preds is not None and refs is not None:
        wer_score = wer_metric.compute(predictions=preds, references=refs)
        cer_score = cer_metric.compute(predictions=preds, references=refs)
        wer_matrix[0, col_idx] = wer_score
        cer_matrix[0, col_idx] = cer_score
        print(f"✨ Kết quả -> WER: {wer_score:.4f} | CER: {cer_score:.4f}")

# 2. Đánh giá 5 Model Parts trên các Test Parts (Ma trận tam giác)
print("\n🕵️‍♂️ BẮT ĐẦU ĐÁNH GIÁ CÁC MODEL CHECKPOINTS (TAM GIÁC)")
print("="*65)
for row_idx, model_part in enumerate(parts):
    checkpoint_p = CHECKPOINTS[model_part]
    if not os.path.exists(checkpoint_p):
        print(f"\n[INFO] Bỏ qua hàng {model_part} vì đường dẫn checkpoint không tồn tại.")
        continue
    # Duyệt cột: Chỉ đánh giá từ Task 1 đến Task hiện tại (row_idx)
    for col_idx in range(row_idx + 1):
        data_part = parts[col_idx]
        print(f"\n📊 Đang đánh giá [Mô hình: {model_part}] trên [Dữ liệu: {data_part}]")
        preds, refs = evaluate_model_on_dataset(checkpoint_p, eval_datasets[data_part])
        if preds is not None and refs is not None:
            wer_score = wer_metric.compute(predictions=preds, references=refs)
            cer_score = cer_metric.compute(predictions=preds, references=refs)
            wer_matrix[row_idx + 1, col_idx] = wer_score
            cer_matrix[row_idx + 1, col_idx] = cer_score
            print(f"✨ Kết quả -> WER: {wer_score:.4f} | CER: {cer_score:.4f}")

In [ ]:
import pandas as pd
import numpy as np
import json

print("\n" + "=*="*20)
print("     KẾT QUẢ MA TRẬN ĐÁNH GIÁ CHÉO CONTINUAL LEARNING (WER / CER)")
print("=*="*20)

N = len(parts)
col_labels = [f"Test Part {i}" for i in range(1, N + 1)]
idx_labels = ["Base Model"] + [f"Model Part {i}" for i in range(1, N + 1)]

# Tạo mảng chứa các chuỗi định dạng gộp "WER / CER"
combined_matrix = []
for r in range(N + 1):
    row_data = []
    for c in range(N):
        wer_val = wer_matrix[r, c]
        cer_val = cer_matrix[r, c]
        
        # Nếu gặp ô trống (NaN), điền kí hiệu "NA" giống thiết kế Canva
        if np.isnan(wer_val) or np.isnan(cer_val):
            row_data.append("NA")
        else:
            row_data.append(f"{wer_val:.4f} / {cer_val:.4f}")
    combined_matrix.append(row_data)

# Chuyển đổi sang Pandas DataFrame để hiển thị tường minh
df_matrix = pd.DataFrame(combined_matrix, index=idx_labels, columns=col_labels)

# In bảng kết quả dạng chuỗi trực quan ra màn hình
print(df_matrix.to_string())